In [20]:
import pandas as pd
import shap
import numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MaxAbsScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, StackingClassifier, VotingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.svm import SVC
from sklearn.compose import ColumnTransformer
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score, precision_score, f1_score, recall_score


In [27]:
cuisine_dataset = pd.read_csv("./Balanced_Cuisine.csv")
output_encode = LabelEncoder()
cuisine_dataset["cuisine"] = output_encode.fit_transform(cuisine_dataset["cuisine"])

cuisine_label = cuisine_dataset["cuisine"]
cuisine_dataset.drop(columns=["cuisine"], inplace=True)
cuisine_dataset.value_counts()


preprocess = ColumnTransformer(transformers=[
    ("scale", MaxAbsScaler(), slice(None)),
])

random_forest = RandomForestClassifier(n_estimators=300, criterion="gini", max_features="sqrt", max_depth=10, max_leaf_nodes=5, random_state=42)
svc_classifier = SVC(C=1.0, kernel="rbf", gamma="scale", probability=True)
kneighbours = KNeighborsClassifier(n_neighbors=5, weights="uniform")
xgboost = XGBClassifier(n_estimators=200, learning_rate=0.01, max_depth=10, min_samples_leaf=5, objective="multi:softmax", num_class = 5)
decision = DecisionTreeClassifier(criterion="entropy", splitter="best", max_depth=10, min_samples_leaf=5)

base_models = [
    ("rnf", random_forest),
    ("svc", svc_classifier),
    ("knn", svc_classifier),
]

meta_model = LogisticRegression(C=1, max_iter=2000)

stacked_model = StackingClassifier(
    estimators=base_models,
    final_estimator=meta_model,
    passthrough=True
)
voting_model = VotingClassifier(
    estimators=base_models,
)
prediction_model = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", svc_classifier)
])
scale = prediction_model.named_steps["preprocess"]
X = cuisine_dataset
y = cuisine_label

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train_scaled = scale.fit_transform(X_train)


prediction_model.fit(X_train, y_train)


prediction = prediction_model.predict(X_test)
prediction


print(f"Precision {precision_score(y_test, prediction, average="macro")}")
print(f"Recall {recall_score(y_test, prediction, average="macro")}")
print(f"F1 {f1_score(y_test, prediction, average="macro")}")
print(f"Accuracy {accuracy_score(y_test, prediction)}")



Precision 0.8137407968700797
Recall 0.8098156606717797
F1 0.8089106035849639
Accuracy 0.8110137672090113
